In [1]:
import numpy as np

import tensorflow as tf
from tensorflow import keras

In [2]:
model = tf.keras.models.load_model('model_2024_hairstyle.keras')

In [4]:
model.export('hairstyle-model')

INFO:tensorflow:Assets written to: hairstyle-model\assets


INFO:tensorflow:Assets written to: hairstyle-model\assets


Saved artifact at 'hairstyle-model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 200, 200, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2775745630032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2775745630800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2775745631568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2775745632336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2775745633488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2775745634064: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [6]:
# Convert the SavedModel to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_saved_model('hairstyle-model/')
tflite_model = converter.convert()

# Save the TFLite model
with open('hairstyle-model.tflite', 'wb') as f_out:
    f_out.write(tflite_model)

#### Question 1
Now convert this model from Keras to TF-Lite format.

What's the size of the converted model?

- 27 Mb
- 43 Mb
- 77 Mb [X]
- 127 Mb

In [8]:
import tensorflow.lite as tflite

In [10]:
interpreter = tflite.Interpreter(model_path='hairstyle-model.tflite')
interpreter.allocate_tensors()

input_index = interpreter.get_input_details()[0]['index']
output_index = interpreter.get_output_details()[0]['index']

input_index, output_index

(0, 13)

#### Question 2
To be able to use this model, we need to know the index of the input and the index of the output.

What's the output index for this model?

- 3
- 7
- 13 [X]
- 24

In [53]:
from io import BytesIO
from urllib import request

from PIL import Image

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img


def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

def rescale_function(image):
    return image * (1. / 255.0)

In [66]:
img = download_image("https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg")

target_size = (200, 200)
img = prepare_image(img, target_size)

x = np.array(img, dtype='float32')
X = np.array([x])
X = rescale_function(X)

X[0][0][0][0]

0.2392157

#### Question 3
Now we need to turn the image into numpy array and pre-process it.

Tip: Check the previous homework. What was the pre-processing we did there?

After the pre-processing, what's the value in the first pixel, the R channel?

- 0.24 [X]
- 0.44
- 0.64
- 0.84

In [69]:
interpreter.set_tensor(input_index, X)
interpreter.invoke()
preds = interpreter.get_tensor(output_index)

preds

array([[0.89377415]], dtype=float32)

#### Question 4
Now let's apply this model to this image. What's the output of the model?

- 0.293
- 0.493
- 0.693
- 0.893 [X]

#### Question 5
Download the base image agrigorev/model-2024-hairstyle:v3. You can do it with docker pull.

So what's the size of this base image?

- 182 Mb
- 382 Mb
- 582 Mb
- 782 Mb [X]

DOCKER FILE

FROM agrigorev/model-2024-hairstyle:v3

RUN pip install https://github.com/alexeygrigorev/tflite-aws-lambda/raw/main/tflite/tflite_runtime-2.14.0-cp310-cp310-linux_x86_64.whl

RUN pip install pillow

RUN pip install numpy==1.23.2

COPY ["lambda_function.py", "./"]

CMD [ "lambda_function.lambda_handler" ]

In [ ]:
# FUNCTION

import tflite_runtime.interpreter as tflite
import numpy as np

from io import BytesIO
from urllib import request

from PIL import Image

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img


def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

def rescale_function(image):
    return image * (1. / 255.0)

def lambda_handler():
    interpreter = tflite.Interpreter(model_path='model_2024_hairstyle_v2.tflite')
    interpreter.allocate_tensors()

    input_index = interpreter.get_input_details()[0]['index']
    output_index = interpreter.get_output_details()[0]['index']

    img = download_image("https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg")

    target_size = (200, 200)
    img = prepare_image(img, target_size)

    x = np.array(img, dtype='float32')
    X = np.array([x])
    X = rescale_function(X)

    interpreter.set_tensor(input_index, X)
    interpreter.invoke()
    preds = interpreter.get_tensor(output_index)

    float_predictions = preds[0].tolist()

    return float_predictions

print(lambda_handler())

#### Question 6
Now let's extend this docker image, install all the required libraries and add the code for lambda.

You don't need to include the model in the image. It's already included. The name of the file with the model is model_2024_hairstyle_v2.tflite and it's in the current workdir in the image (see the Dockerfile above for the reference). The provided model requires the same preprocessing for images regarding target size and rescaling the value range than used in homework 8.

Now run the container locally.

Score this image: https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg

What's the output from the model?

- 0.229
- 0.429 [X]
- 0.629
- 0.829